#### READ THE GEO LOCATIONS DATA FROM VOLUME

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/GeoLocations/'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'GEO_LOCATIONS'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
AUDIT_TABLE  = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
geo_locations_schema = """geolocation_zip_code_prefix INT, 
                 geolocation_lat STRING, 
                 geolocation_lng STRING, 
                 geolocation_city STRING, 
                 geolocation_state STRING"""

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

geo_locations_df = (
            spark.read.format('csv')
                 .option('header', True)
                 .schema(geo_locations_schema)
                 .load(f'{BASE_DIR}')
        )
geo_locations_trans_df = geo_locations_df.select("*", "_metadata.file_name", "_metadata.file_path").withColumn('load_timestamp', current_timestamp())
geo_locations_trans_df.printSchema()

In [0]:
geo_locations_trans_df.limit(1).display()

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.GEO_LOCATIONS`

In [0]:
def write_audit(run_id, status, start_time, rows=None, error=None):
    end = datetime.now()
    row = (run_id, "GEO_LOCATIONS_LOAD", "BRONZE", BASE_DIR, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
    (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

# ---------- Load with audit ----------
run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
    geo_locations_df = (spark.read.format("csv")
                             .option("header", True)
                             .schema(geo_locations_schema)
                             .load(BASE_DIR))

    geo_locations_trans_df = (geo_locations_df
        .select("*", "_metadata.file_name", "_metadata.file_path")
        .withColumn("load_timestamp", F.current_timestamp()))

    geo_locations_trans_df.write.mode("append").format("delta").saveAsTable(TARGET)

    metrics = DeltaTable.forName(spark, TARGET).history(1).select("operationMetrics").first()[0]
    rows_written = int(metrics.get("numOutputRows", 0))

except Exception:
    status, err = "FAILED", traceback.format_exc()
    raise
finally:
    write_audit(run_id, status, start_time, rows_written, err)

# display(spark.table(AUDIT_TABLE).orderBy(F.desc("start_time")).limit(20))

In [0]:
dbutils.notebook.exit('GEO LOCATIONS BRONZE INGESTION IS LOADED')